# HELYUM ATOMU İÇİN HARTREE-FOCK METHODU



$$
\textbf{1) Gaussian Baz Fonksiyonu}
$$

$$
\phi(\mathbf{r}) = \sum_i d_i e^{-\alpha_i r^2}
$$

$$
\textbf{2) Overlap (Örtüşme) Matrisi}
$$

$$
S_{ij} = \int \phi_i \phi_j \, d\tau
$$

$$
S_{ij} = d_i d_j \left( \frac{\pi}{a + b} \right)^{3/2}
$$

$$
\textbf{3) Kinetik Enerji Matrisi}
$$

$$
T_{ij} = -\frac{1}{2} \int \phi_i \nabla^2 \phi_j \, d\tau
$$

$$
T_{ij} = d_i d_j \frac{3ab\pi^{3/2}}{(a+b)^{5/2}}
$$

$$
\textbf{4) Çekirdek Çekim Potansiyeli}
$$

$$
V_{ij} = \int \phi_i \left(-\frac{Z}{r}\right) \phi_j d\tau
$$

$$
V_{ij} = - d_i d_j \frac{2\pi Z}{a+b}
$$

$$
\textbf{5) Elektron-Elektron İtme İntegrali (ERI)}
$$

$$
(\mu\nu|\lambda\sigma)
=
\int\int
\frac{\phi_\mu(1)\phi_\nu(1)\phi_\lambda(2)\phi_\sigma(2)}
{r_{12}}
d\tau_1 d\tau_2
$$

$$
(\mu\nu|\lambda\sigma)
=
\frac{2\pi^{5/2}}
{(a+b)(c+e)\sqrt{a+b+c+e}}
$$

$$
\textbf{6) Fock Matrisi}
$$

$$
F_{\mu\nu} = H_{\mu\nu}^{core} + G_{\mu\nu}
$$

$$
H^{core}_{\mu\nu} = T_{\mu\nu} + V_{\mu\nu}
$$

$$
G_{\mu\nu} =
\sum_{\lambda\sigma}
D_{\lambda\sigma}
\left[
(\mu\nu|\lambda\sigma)
-
\frac{1}{2}
(\mu\lambda|\nu\sigma)
\right]
$$

$$
\textbf{7) Roothaan Denklemi}
$$

$$
F C = S C \varepsilon
$$

$$
\textbf{8) Yoğunluk Matrisi}
$$

$$
D_{\mu\nu} = 2 C_\mu C_\nu
$$

$$
\textbf{9) Toplam Hartree-Fock Enerjisi}
$$

$$
E =
\frac{1}{2}
\sum_{\mu\nu}
D_{\mu\nu}
\left(
H_{\mu\nu}^{core}
+
F_{\mu\nu}
\right)
$$
"""

In [30]:
import numpy as np
from scipy.linalg import eigh

def solve_helium_hf_ev():
    # 1. Sabitler ve Parametreler 
    # Alpha: Gaussian genişlikleri, d: Kontraksiyon katsayıları
    alpha = np.array([6.36242139, 1.15892300, 0.31364979])
    d = np.array([0.15432897, 0.53532814, 0.44463454])
    N = len(alpha)
    Z = 2  # Helyum çekirdek yükü
    HARTREE_TO_EV = 27.211386  
    # 2. Matrislerin Hazırlanması (S, T, V)
    S = np.zeros((N, N)) # Örtüşme (Overlap)
    T = np.zeros((N, N)) # Kinetik Enerji
    V = np.zeros((N, N)) # Çekirdek Çekim (Potential)

    for i in range(N):
        for j in range(N):
            a = alpha[i]
            b = alpha[j]
            # Gaussian integralleri için analitik formüller
            S[i, j] = d[i] * d[j] * (np.pi / (a + b))**1.5
            T[i, j] = d[i] * d[j] * 3 * a * b * np.pi**1.5 / (a + b)**2.5
            V[i, j] = -d[i] * d[j] * 2 * np.pi * Z / (a + b)

    H_core = T + V

    # İki elektron itme integrali 
    def eri_func(a, b, c, e):
        return 2 * np.pi**2.5 / ((a + b) * (c + e) * np.sqrt(a + b + c + e))

    # 3. SCF (Self-Consistent Field) Döngüsü
    D = np.zeros((N, N))  # Başlangıç yoğunluk matrisi (
    threshold = 1e-8      # Yakınsama kriteri
    E_old = 0.0
    
    print(f"{'İterasyon':<10} | {'Enerji (Hartree)':<18} | {'Enerji (eV)':<15}")
    print("-" * 50)

    for iteration in range(100):
        # Fock Matrisi Oluşturma (F = H_core + G)
        G = np.zeros((N, N))
        for i in range(N):
            for j in range(N):
                for k in range(N):
                    for l in range(N):
                        # Coulomb ve Exchange etkileşimleri
                       
                        val = d[i]*d[j]*d[k]*d[l] * eri_func(alpha[i], alpha[j], alpha[k], alpha[l])
                        G[i, j] += D[k, l] * (val - 0.5 * val)

        F = H_core + G
        
        # Roothaan Denklemleri: F C = E S C
        epsilon, C = eigh(F, S)
        
        # Helyumda 2 elektron vardır, ikisi de en düşük enerjili orbitale (n=0) yerleşir
        C_lowest = C[:, 0]
        
        D = 2 * np.outer(C_lowest, C_lowest)
        
        # Toplam Enerji Hesabı (E = 0.5 * sum(D * (H_core + F)))
        E_total = 0.5 * np.sum(D * (H_core + F))
        E_ev = E_total * HARTREE_TO_EV
        
        print(f"{iteration+1:<10} | {E_total:<18.8f} | {E_ev:<15.4f}")
        
        if abs(E_total - E_old) < threshold:
            break
        E_old = E_total

    return E_total, E_ev


final_h, final_ev = solve_helium_hf_ev()

print("-" * 50)
print(f"Helyum Taban Durum Enerjisi (Final):")
print(f"Hartree: {final_h:.6f} Ha")
print(f"eV:      {final_ev:.4f} eV")
print("-" * 50)

İterasyon  | Enerji (Hartree)   | Enerji (eV)    
--------------------------------------------------
1          | -3.93731122        | -107.1397      
2          | -2.69863391        | -73.4336       
3          | -2.84824481        | -77.5047       
4          | -2.80790914        | -76.4071       
5          | -2.81844803        | -76.6939       
6          | -2.81566683        | -76.6182       
7          | -2.81639896        | -76.6381       
8          | -2.81620611        | -76.6329       
9          | -2.81625690        | -76.6343       
10         | -2.81624352        | -76.6339       
11         | -2.81624704        | -76.6340       
12         | -2.81624611        | -76.6340       
13         | -2.81624636        | -76.6340       
14         | -2.81624629        | -76.6340       
15         | -2.81624631        | -76.6340       
16         | -2.81624631        | -76.6340       
--------------------------------------------------
Helyum Taban Durum Enerjisi (Final):
Hartree: -2